# Lab 1 · Self-attention (simple)
The word **“she”** means nothing alone: you need to know *who*.
Self-attention lets each word **look at the other words and borrow from them**.

It is just three steps: **score → share → blend**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.set_printoptions(precision=3, sci_mode=False)
torch.set_grad_enabled(False)   # no training in this lab

## 1 · Words as numbers
Each word gets 2 numbers: `[person, thing]`.

In [ ]:
words = ['Asha', 'exam', 'she']
X = torch.tensor([[2., 0.],    # Asha: a person
                  [0., 2.],    # exam: a thing
                  [1., 0.]])   # she : a person, but who?
print(X)

## 2 · Score: how alike is *she* to each word?
Dot product: multiply matching numbers, add them up. Alike → big number.

In [ ]:
she = X[2]
scores = None        # TODO: X @ she
print(scores)

## 3 · Share: turn scores into percentages
Softmax makes the scores positive and adds them up to 1.

In [ ]:
weights = None   # TODO: torch.softmax(scores, dim=0)
for w, a in zip(words, weights):
    print(f'she → {w:4} {a:.0%}')

## 4 · Blend: mix the words using those percentages
*she* takes 67% of Asha + 9% of exam + 24% of herself.

In [ ]:
she_new = None   # TODO: weights @ X
print('she before:', X[2])
print('she after :', she_new)

That’s self-attention for one word. ✓

## 5 · All words at once
Do the same for every word in one go. The score table is **words × words** (3 × 3):
each row is one word scoring all 3 words.

In [ ]:
scores  = None   # TODO: X @ X.T
weights = None   # TODO: softmax along each row (dim=1)
out     = None   # TODO: weights @ X
print(weights)
print(out)

Row 3 is *she*: the same numbers as steps 2–4.

## 6 · Query, Key, Value
So far each word used **its own vector** for everything. Real models make **3 versions** of each word,
each with its own learned layer:

| | Meaning | Library analogy |
| --- | --- | --- |
| **Query** Q | what I’m looking for | my question |
| **Key** K | what I offer | the label on a book |
| **Value** V | what I hand over | what’s inside the book |

Score with Q and K, blend the V’s. Same three steps.

In [ ]:
torch.manual_seed(0)
W_Q, W_K, W_V = nn.Linear(2, 2, bias=False), nn.Linear(2, 2, bias=False), nn.Linear(2, 2, bias=False)
Q, K, V = W_Q(X), W_K(X), W_V(X)          # random now; training would learn them

weights = torch.softmax(Q @ K.T, dim=1)   # score + share
out     = weights @ V                     # blend
print(weights)

## 7 · The full formula
One last detail: divide the scores by **√d** (d = how many numbers each word has, here 2).
Real words have hundreds of numbers, which makes scores huge; dividing keeps softmax from putting 100% on one word.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d}}\right) V$$

In [ ]:
def attention(Q, K, V):
    d = K.shape[-1]
    weights = None   # TODO: softmax(Q @ K.mT / √d) along the last dim
    return weights @ V

mine    = attention(Q, K, V)
pytorch = F.scaled_dot_product_attention(Q, K, V)
print(mine)
print('same as PyTorch?', torch.allclose(mine, pytorch))

## Recap
1. **Score**: each word’s query · every word’s key
2. **Share**: softmax → percentages that add up to 1
3. **Blend**: mix the values with those percentages

That’s self-attention. A Transformer stacks this layer many times.

**Try it:** change *she* to `[1., 1.]` in step 1 (a bit person, a bit thing) and re-run. Who does she look at now?